In [ ]:
import os
import sys
from pathlib import Path
import duckdb
import toml

from loguru import logger
# from horse_logic.motherduck import DuckDBManager

In [ ]:
def stdout_filter(record):
    return record["level"].name == "INFO" and "ECHO" in record["message"]

def customise_logger(logger_filename="DuckDBManager.log", logger_size=10, filter_func=stdout_filter, level="INFO"):
    try:
        logger.remove(0) # Remove default output to sys.stderr
        logger.add(sys.stdout, filter=filter_func, level=level)     # Add sys.stdout logging of select messages
        logger.add(logger_filename, rotation=f"{logger_size} MB", level=level)  # logs will be written to file.log
        logger.info(f"ECHO: Logging to {logger_filename}: size = {logger_size} MB")
        logger.info("ECHO: Limited logging to stdout using `stdout_filter`")
    except Exception as e:
        logger.error(f"Error with customising logger: {e}")

In [ ]:
customise_logger()

In [ ]:
logger.info(f"DuckDB version: {duckdb.__version__}")

In [ ]:
SECRETS_TOML = Path.cwd().parent / ".secrets.toml"

In [ ]:
def set_md_token(secrets_file):
    """
    Load secrets from a file and set the motherduck_token environment variable.
    """
    if "motherduck_token" not in os.environ:
        try:
            secrets = toml.load(secrets_file)
            os.environ["motherduck_token"] = secrets["MD_TOKEN"]
            logger.info(f"ECHO: Secrets loaded from {secrets_file} and MotherDuck environment variable set")
        except Exception as e:
            logger.error(f"Failed to load secrets: {e}")
            raise
    else:
        logger.info("ECHO: MD environment variable already set.")

In [ ]:
set_md_token(SECRETS_TOML)

In [ ]:
con_md = duckdb.connect("md:")

In [ ]:
con_md.sql("SHOW DATABASES;")

In [ ]:
def check_local_duckdb_database():
    """
    Checks if a local DuckDB database exists. It returns the path to the database if it exists, and None otherwise.
    """
    DATA_DIR = Path("../data")
    EXPERIMENT_TYPE = "RPE"  # or CB
    assert DATA_DIR.exists()
    local_ddb = DATA_DIR / f"Experiments_{EXPERIMENT_TYPE}_2023_Q4.ddb"  # DuckDB database name

    logger.info(f"Database file: {local_ddb.resolve()}")

    if local_ddb.exists():
        return local_ddb
    else:
        return None

In [ ]:
local_ddb = check_local_duckdb_database()

In [ ]:
local_ddb.stem

In [ ]:
attach_local_ddb = f"ATTACH '{str(local_ddb)}' AS local_{local_ddb.stem.replace('-', '_')};"

In [ ]:
con_md.sql(attach_local_ddb)

In [ ]:
con_md.sql("SHOW DATABASES;")

In [ ]:

ddb = DuckDBManager(db_path=str(local_ddb), secrets_file=None, logger=logger, dry_run=True)

# con = duckdb.connect(str(local_ddb.resolve()))

In [ ]:
ddb.connect()

In [ ]:
ddb.execute_sql("SHOW ALL TABLES;")

In [ ]:
ddb.execute_sql("SELECT * FROM Experiments")

In [ ]:
ddb.execute_sql("PRAGMA database_list;")

In [ ]:
# def standardise_db_name(name_in):
#     return name_in.replace("-", "_").lower()

In [ ]:
ddb.get_motherduck_ui_version()

In [ ]:
ddb.dry_run

In [ ]:
ddb.copy_ddb_to_motherduck(str(local_ddb), remote_db_name="test_new_class")

In [ ]:
# with DuckDBManager(db_path=str(local_ddb), secrets_file=SECRETS_TOML, logger=logger) as test_manager:
#     md_db_version_info = test_manager.get_motherduck_db_version()
#     md_ui_version_info = test_manager.get_motherduck_ui_version()

In [ ]:
# md_db_version_info, md_ui_version_info

In [ ]:
with DuckDBManager(db_path=str(local_ddb), secrets_file=SECRETS_TOML, logger=logger) as db_manager:
    db_manager.copy_ddb_to_motherduck(local_ddb, remote_db_name="test_class_approach")

In [ ]:
# Code for working with shared database

# con_md_share = duckdb.connect()

In [ ]:
# -- Run this snippet to attach database
# ATTACH 'md:_share/experiments_rpe_2023_q4/e7b8df3a-fc41-4578-9ce0-553b30685626'


# con_md_share.sql("ATTACH 'md:_share/clouddb/cbe96e98-c6b3-4e13-86b4-c3b42cf95aa6' AS md_ddb")
# con_md_share.sql("USE md_ddb;")

# Learn about sharing here: https://motherduck.com/docs/key-tasks/querying-a-shared-motherduck-database.

In [ ]:
# -- Run this snippet to attach database
# ATTACH 'md:_share/experiments_rpe_2023_q4/e7b8df3a-fc41-4578-9ce0-553b30685626'


# con_md_share.sql("ATTACH 'md:_share/clouddb/cbe96e98-c6b3-4e13-86b4-c3b42cf95aa6' AS md_ddb")
# con_md_share.sql("USE md_ddb;")

# Learn about sharing here: https://motherduck.com/docs/key-tasks/querying-a-shared-motherduck-database.

In [ ]:
# con_md_share.sql("SELECT * FROM Experiments")

In [ ]:
# con_md_share.close()